# Day 15: Exploring Pretrained AI Models with Hugging Face

## Objective

Explore the Hugging Face Model Hub and experiment with pretrained models for text generation, sentiment analysis, and text summarization.

## Learning Objectives

- Understand the purpose of the Hugging Face Model Hub.
- Explore model cards and identify models suitable for different tasks.
- Run three pretrained AI models using Python and Hugging Face Transformers.
- Examine and evaluate actual model outputs.
- Compare model results using task-specific evaluation criteria.

## Tools and Environment

- Google Colab
- Python
- Hugging Face Model Hub
- Hugging Face Transformers
- PyTorch


## 1. Exploring the Hugging Face Model Hub

The Hugging Face Model Hub is a platform where developers and researchers share pretrained machine learning models. Models are available for tasks such as text generation, sentiment analysis, summarization, translation, and question answering.

A model repository typically contains a model card, configuration files, tokenizer files, and model weights. The model card describes the model's intended use, supported tasks, limitations, and instructions for use.

### Model Hub Exploration

Visit the following pages and inspect the available models.

1. [Hugging Face Model Hub](https://huggingface.co/models)
2. [Text Generation Models](https://huggingface.co/models?pipeline_tag=text-generation)
3. [Text Classification Models](https://huggingface.co/models?pipeline_tag=text-classification)
4. [Summarization Models](https://huggingface.co/models?pipeline_tag=summarization)

### Selected Models

| Task | Selected Model | Model Repository |
|---|---|---|
| Text Generation | Qwen2.5-0.5B-Instruct | Qwen/Qwen2.5-0.5B-Instruct |
| Sentiment Analysis | Twitter-RoBERTa-Sentiment-Latest | cardiffnlp/twitter-roberta-base-sentiment-latest |
| Summarization | FLAN-T5-Base | google/flan-t5-base |

### What to Examine in a Model Card

- Intended use and supported tasks
- Model architecture or model type
- Usage instructions and example code
- Limitations and potential failure cases
- Model files and download information

### Exploration Notes

After visiting the Model Hub, record two or three observations about how models are organized, how model cards help users select models, and what information is provided about model usage and limitations.

In [26]:
%pip -q install transformers sentencepiece

import time
import logging
import torch
import pandas as pd
from transformers import pipeline
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)

print("PyTorch version:", torch.__version__)
print("CUDA GPU available:", torch.cuda.is_available())

device = 0 if torch.cuda.is_available() else -1
print("Device selected:", "GPU" if device == 0 else "CPU")

PyTorch version: 2.11.0+cpu
CUDA GPU available: False
Device selected: CPU


## 2. Experiment 1 — Text Generation

### Objective

Generate a relevant response to a prompt using a pretrained instruction-tuned language model.

### Selected Model

Qwen2.5-0.5B-Instruct

This model is an instruction-tuned language model. It can be used to generate responses to prompts and follow simple natural-language instructions.

### Input

Explain three practical ways artificial intelligence can improve education, using simple language and examples.

### Evaluation Criteria

- Relevance to the prompt
- Coherence and readability
- Following the instruction to provide three examples
- Inference time

The generated output will be evaluated after the model has been executed.

In [31]:
text_generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=device
)

generation_prompt = (
    "Explain three practical ways artificial intelligence "
    "can improve education. Use simple language and "
    "give one example for each."
)

messages = [
    {"role": "user", "content": generation_prompt}
]

formatted_prompt = text_generator.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

start_time = time.time()

generation_result = text_generator(
    formatted_prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False,
    pad_token_id=text_generator.tokenizer.eos_token_id
)

generation_time = time.time() - start_time
generated_text = generation_result[0]["generated_text"]

print("Model: Qwen2.5-0.5B-Instruct")
print("Input prompt:", generation_prompt)
print("\nGenerated output:\n", generated_text)
print(f"\nInference time: {generation_time:.2f} seconds")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model: Qwen2.5-0.5B-Instruct
Input prompt: Explain three practical ways artificial intelligence can improve education. Use simple language and give one example for each.

Generated output:
 Sure! Here are three practical ways AI can improve education:

1. **Personalized Learning Paths**: AI can analyze student data to create personalized learning paths tailored to their individual needs and pace of learning. For instance, if a student is struggling with math, an AI could suggest resources like video tutorials or interactive exercises that match the student's current level of understanding.

2. **Adaptive Assessment Tools**: AI can use machine learning algorithms to adapt assessments based on students' performance over time. This means that as students progress through different levels of difficulty in a subject, the assessment becomes more challenging but still manageable

Inference time: 54.54 seconds


## 3. Experiment 2 — Sentiment Analysis

### Objective
Classify a text review according to its sentiment using a pretrained text-classification model.

### Input
"The product quality is excellent, and the delivery was faster than expected."

### Evaluation Criteria
- Whether the predicted label matches the expected sentiment
- Confidence score associated with the prediction
- Inference time

The predicted label and confidence score will be recorded from the actual model output.


In [32]:

import time
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device
)

review = (
    "The product quality is excellent, and the delivery "
    "was faster than expected."
)

start_time = time.time()
sentiment_result = sentiment_analyzer(review)
sentiment_time = time.time() - start_time

predicted_label = sentiment_result[0]["label"]
sentiment_confidence = sentiment_result[0]["score"]

print("Model: Twitter-RoBERTa-Sentiment-Latest")
print("Input review:", review)
print("Predicted sentiment:", predicted_label)
print(f"Confidence score: {sentiment_confidence:.4f}")
print(f"Inference time: {sentiment_time:.2f} seconds")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model: Twitter-RoBERTa-Sentiment-Latest
Input review: The product quality is excellent, and the delivery was faster than expected.
Predicted sentiment: positive
Confidence score: 0.9846
Inference time: 0.16 seconds



## 4. Experiment 3 — Text Summarization

### Objective

Condense a longer passage into a shorter version while preserving its main ideas.

### Selected Model

FLAN-T5-Base

FLAN-T5 is an instruction-tuned text-to-text model. It can accept a text instruction and produce a corresponding text output.

### Input

A passage explaining the educational benefits of AI, its risks, and the importance of human guidance.

### Evaluation Criteria

- Preservation of the main ideas
- Conciseness
- Readability
- Inference time

The generated summary will be compared with the original passage to identify important information that has been retained or omitted.


In [33]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "google/flan-t5-base"

# Load the tokenizer and pretrained model
tokenizer = AutoTokenizer.from_pretrained(model_id)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
summarizer_model = summarizer_model.to(torch_device)
summarizer_model.eval()

passage = """
Artificial Intelligence is increasingly being used in education.
It can help teachers identify areas where students need additional
support and provide learners with personalized practice activities.
AI-powered tools can also assist with feedback, language learning,
and access to educational resources. However, these tools should
be used responsibly. Students and teachers must consider privacy,
the accuracy of generated information, and the risk of relying
too heavily on automated systems. Human guidance remains important
for effective learning.
"""

summarization_prompt = (
    "Summarize the main benefits, risks, and importance of "
    "human guidance in the following passage in 2-3 sentences:\n" + passage
)

# Tokenize the input and move it to the selected device
inputs = tokenizer(
    summarization_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(torch_device)

# Generate the summary
start_time = time.time()

with torch.inference_mode():
    summary_ids = summarizer_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

summary_time = time.time() - start_time
summary_text = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Model: FLAN-T5-Base")
print("Original passage:\n", passage)
print("\nGenerated summary:\n", summary_text)
print(f"\nInference time: {summary_time:.2f} seconds")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Model: FLAN-T5-Base
Original passage:
 
Artificial Intelligence is increasingly being used in education.
It can help teachers identify areas where students need additional
support and provide learners with personalized practice activities.
AI-powered tools can also assist with feedback, language learning,
and access to educational resources. However, these tools should
be used responsibly. Students and teachers must consider privacy,
the accuracy of generated information, and the risk of relying
too heavily on automated systems. Human guidance remains important
for effective learning.


Generated summary:
 Use human guidance to help students learn.

Inference time: 1.53 seconds



## 5. Comparative Analysis

The experiments use three different pretrained models for three distinct natural language processing tasks. Each output should be evaluated using criteria appropriate to its task.

| Experiment | Model | Evaluation Criteria |
|---|---|---|
| Text Generation | Qwen2.5-0.5B-Instruct | Relevance, coherence, instruction following |
| Sentiment Analysis | Twitter-RoBERTa-Sentiment-Latest | Correct sentiment label and confidence |
| Summarization | FLAN-T5-Base | Coverage of key ideas, conciseness, readability |

### Observations

- Text generation produces a continuation or response based on the supplied prompt.
- Sentiment analysis classifies the emotional polarity of the input text.
- Summarization condenses a passage while attempting to preserve its central ideas.
- Inference time can vary with model size, hardware, runtime initialization, and input length.

The three models perform different tasks, so their raw outputs cannot be ranked directly against one another. The best model for each task should be identified from the observed output and its evaluation criteria.


In [34]:

import pandas as pd

comparison = pd.DataFrame([
    {
        "Task": "Text Generation",
        "Model": "Qwen2.5-0.5B-Instruct",
        "Observed Result": generated_text,
        "Inference Time (seconds)": round(generation_time, 2)
    },
    {
        "Task": "Sentiment Analysis",
        "Model": "Twitter-RoBERTa-Sentiment-Latest",
        "Observed Result": (
            f"{predicted_label} "
            f"(confidence: {sentiment_confidence:.4f})"
        ),
        "Inference Time (seconds)": round(sentiment_time, 2)
    },
    {
        "Task": "Summarization",
        "Model": "FLAN-T5-Base",
        "Observed Result": summary_text,
        "Inference Time (seconds)": round(summary_time, 2)
    }
])

comparison


,Task,Model,Observed Result,Inference Time (seconds)
0,Text Generation,Qwen2.5-0.5B-Instruct,Sure! Here are three practical ways AI can imp...,54.54
1,Sentiment Analysis,Twitter-RoBERTa-Sentiment-Latest,positive (confidence: 0.9846),0.16
2,Summarization,FLAN-T5-Base,Use human guidance to help students learn.,1.53



## 6. Results and Conclusion

### Results

Three pretrained models were tested for text generation, sentiment analysis, and summarization. Qwen2.5-0.5B-Instruct generated a response to a prompt about improving education with AI. Twitter-RoBERTa-Sentiment-Latest classified the product review as positive with a confidence score of 0.9846. FLAN-T5-Base generated a short summary: "Use human guidance to help students learn."

The recorded inference times were 54.54 seconds for text generation, 0.16 seconds for sentiment analysis, and 1.53 seconds for summarization. These measurements reflect the current runtime and may vary across executions.

### Best-Performing Model

Each model performed a different task, so their outputs cannot be directly ranked against one another. The sentiment analysis result was clearly interpretable for the chosen review. The text-generation output should be evaluated for relevance, coherence, and instruction following. The summarization output was relevant but too brief to preserve all the key points of the original passage.

The most suitable model therefore depends on the task-specific requirements, output quality, inference time, and computational resources.

### Conclusion

The experiments demonstrate how pretrained models can be applied to different language-processing tasks without training a model from scratch. They also show that generated outputs require critical evaluation: a confident sentiment prediction does not guarantee correctness, and a concise summary may omit important information.

The results provide practical experience with loading pretrained models from the Hugging Face Model Hub, running inference in Google Colab, and comparing model outputs using task-specific criteria.